In [3]:
# Test 1: Check Python environment
import sys
print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")

Python version: 3.11.11 | packaged by conda-forge | (main, Dec  5 2024, 14:17:24) [GCC 13.3.0]
Python path: /opt/conda/bin/python3.11


In [2]:
# Test 2: Check CUDA
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")



PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4080 Laptop GPU
CUDA memory allocated: 0.00 GB


In [3]:

# Test 3: Check key imports
import numpy as np
import pandas as pd
import transformers
import sentence_transformers
import llama_index
import langchain

print(f"\nNumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Sentence Transformers: {sentence_transformers.__version__}")
print(f"LangChain: {langchain.__version__}")


NumPy: 1.26.4
Pandas: 2.2.1
Transformers: 4.40.2
Sentence Transformers: 3.1.1
LangChain: 1.2.3


In [7]:
import httpx
import json
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

# Test all services
print("Testing all services...")
services = {
    "LLM Server": "http://localhost:8001/health",
    "Embedding Server": "http://localhost:8002/health",
    "Qdrant": "http://localhost:6333"
}

for name, url in services.items():
    try:
        if name == "Qdrant":
            resp = httpx.get(url, timeout=5)
            status = "✓" if resp.status_code == 200 else "✗"
        else:
            resp = httpx.get(url, timeout=5)
            data = resp.json()
            status = "✓" if data.get("status") == "healthy" else "✗"
        print(f"{name}: {status}")
    except Exception as e:
        print(f"{name}: ✗ ({str(e)[:50]})")


Testing all services...
LLM Server: ✓
Embedding Server: ✓
Qdrant: ✓


In [13]:
from datasets import load_dataset

eval_data = load_dataset("explodinggradients/fiqa", "ragas_eval_v3")
eval_data = eval_data["baseline"]
with open('/workspace/data/fiqa/eval_data.jsonl', 'w') as f:
    for l in eval_data:
        f.write(json.dumps(l) + '\n')

corpus = load_dataset("vibrantlabsai/fiqa", "corpus")
corpus = corpus["corpus"]
with open('/workspace/data/fiqa/corpus.jsonl', 'w') as f:
    for l in corpus:
        f.write(json.dumps(l) + '\n')

main = load_dataset("vibrantlabsai/fiqa", "main")
test_data = main["test"]
with open('/workspace/data/fiqa/test.jsonl', 'w') as f:
    for l in test_data:
        f.write(json.dumps(l) + '\n')
train_data = main["train"]
with open('/workspace/data/fiqa/train.jsonl', 'w') as f:
    for l in train_data:
        f.write(json.dumps(l) + '\n')
validation_data = main["validation"]
with open('/workspace/data/fiqa/validation.jsonl', 'w') as f:
    for l in validation_data:
        f.write(json.dumps(l) + '\n')

In [8]:
#  Setup Qdrant
print("\nSetting up Qdrant...")
client = QdrantClient(host="localhost", port=6333, timeout=10)

collection_name = "fiqa_documents"
if not client.collection_exists(collection_name):
    print("creating collection...")
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )


Setting up Qdrant...
creating collection...


In [9]:
print("\nLoading eval_data data...")
with open('/workspace/data/fiqa/eval_data.jsonl', 'r') as f:
    data = [json.loads(l) for l in f]
df = pd.DataFrame(data)
print(f"Loaded {len(df)} samples")


Loading eval_data data...
Loaded 30 samples


In [10]:
# Generate embeddings and upload
print("Generating embeddings...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Generating embeddings...


/opt/conda/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:
points = []
for idx, row in df.iterrows():
    contexts = row['retrieved_contexts']
    
    if isinstance(contexts, list):
        context_text = " ".join(str(ctx) for ctx in contexts)
    else:
        context_text = str(contexts)

    embedding = embedding_model.encode(context_text)
    points.append(PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={
            # "user_input": row['user_input'],
            # "reference": row['reference'],
            # "response": row['response'],
            "retrieved_contexts": contexts if isinstance(contexts, list) else [contexts]
        }
    ))

# Upload to Qdrant
client.upsert(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} documents")

Uploaded 30 documents


In [12]:
# Get collection info
collection_info = client.get_collection(collection_name)
print(f"Points: {collection_info.points_count}")
print(f"Vectors: {collection_info.config.params.vectors.size}")

Points: 30
Vectors: 384


In [13]:
# Test retrieval
print("\nTesting retrieval...")

# Test query
test_query = df['user_input'][1]
print(f"\nQuery: {test_query}")

query_embedding = embedding_model.encode(test_query).tolist()

results = client.query_points(
    collection_name=collection_name,
    query=query_embedding,
    limit=3,
    with_payload=True 
)

print(f"Found {len(results.points)} results:")
for i, point in enumerate(results.points):
    print(f"\n{i+1}. Score: {point.score:.3f}")
    print(f"   ID: {point.id}")
    print(f"   retrieved_contexts: {point.payload.get('retrieved_contexts', 'No text')[:150]}...")
    # print(f"   DocID: {point.payload.get('retrieved_contexts', 'N/A')}")



Testing retrieval...

Query: Can I send a money order from USPS as a business?
Found 3 results:

1. Score: 0.659
   ID: 1
   retrieved_contexts: ['Sure you can.  You can fill in whatever you want in the From section of a money order, so your business name and address would be fine. The price only includes the money order itself.  You can hand deliver it yourself if you want, but if you want to mail it, you\'ll have to provide an envelope and a stamp. Note that, since you won\'t have a bank record of this payment, you\'ll want to make sure you keep other records, such as the stub of the money order.  You should probably also ask the contractor to give you a receipt."Lets say you owed me $123.00 an wanted to mail me a check. I would then take the check from my mailbox an either take it to my bank, or scan it and deposit it via their electronic interface. Prior to you mailing it you would have no idea which bank I would use, or what my account number is. In fact I could have multiple ban

In [15]:
# Test LLM generation
print("\nTesting LLM with RAG...")
context = "\n".join([f"- {r.payload['retrieved_contexts']}" for r in results.points])
prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {test_query}

Answer:"""

resp = httpx.post(
    "http://localhost:8001/generate",
    json={"prompt": prompt, "max_tokens": 150, "temperature": 0.7},
    timeout=120.0
)

if resp.status_code == 200:
    answer = resp.json()['text']
    print(f"\nGenerated answer: {answer}")
else:
    print(f"LLM error: {resp.status_code}")

print("\n✅ Setup complete! Ready for RAGAS evaluation.")


Testing LLM with RAG...

Generated answer: Based on the following context, answer the question.

Context:
- ['Sure you can.  You can fill in whatever you want in the From section of a money order, so your business name and address would be fine. The price only includes the money order itself.  You can hand deliver it yourself if you want, but if you want to mail it, you\'ll have to provide an envelope and a stamp. Note that, since you won\'t have a bank record of this payment, you\'ll want to make sure you keep other records, such as the stub of the money order.  You should probably also ask the contractor to give you a receipt."Lets say you owed me $123.00 an wanted to mail me a check. I would then take the check from my mailbox an either take it to my bank, or scan it and deposit it via their electronic interface. Prior to you mailing it you would have no idea which bank I would use, or what my account number is. In fact I could have multiple bank accounts, so I could decide which o

In [ ]:
import evaluate
from ragas import evaluate as ragas_evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevance,
    context_recall,
    context_precision,
    answer_correctness
)
import warnings
warnings.filterwarnings('ignore')

#  ========== ROUGE ==========
print("\n📈 ROUGE Scores...")
rouge = evaluate.load("rouge")

In [ ]:


rouge_scores = []
for _, row in eval_df.iterrows():
    scores = rouge.compute(
        predictions=[row['generated_answer']],
        references=[row['ground_truth']]
    )
    rouge_scores.append({
        "rouge1": scores['rouge1'],
        "rouge2": scores['rouge2'],
        "rougeL": scores['rougeL'],
        "rougeLsum": scores['rougeLsum']
    })

rouge_df = pd.DataFrame(rouge_scores)
print(f"Average ROUGE scores:")
print(f"  ROUGE-1:  {rouge_df['rouge1'].mean():.3f}")
print(f"  ROUGE-2:  {rouge_df['rouge2'].mean():.3f}")
print(f"  ROUGE-L:  {rouge_df['rougeL'].mean():.3f}")
print(f"  ROUGE-Lsum: {rouge_df['rougeLsum'].mean():.3f}")

# ========== BERTScore ==========
print("\n📊 BERTScore...")
bertscore = evaluate.load("bertscore")

bert_scores = []
for _, row in eval_df.iterrows():
    scores = bertscore.compute(
        predictions=[row['generated_answer']],
        references=[row['ground_truth']],
        lang="en",
        model_type="microsoft/deberta-xlarge-mnli"
    )
    bert_scores.append({
        "precision": scores['precision'][0],
        "recall": scores['recall'][0],
        "f1": scores['f1'][0]
    })

bert_df = pd.DataFrame(bert_scores)
print(f"Average BERTScore:")
print(f"  Precision: {bert_df['precision'].mean():.3f}")
print(f"  Recall:    {bert_df['recall'].mean():.3f}")
print(f"  F1:        {bert_df['f1'].mean():.3f}")

# ========== RAGAS ==========
print("\n🎯 RAGAS Metrics...")

# Prepare dataset for RAGAS
ragas_dataset = {
    "question": eval_df["question"].tolist(),
    "answer": eval_df["generated_answer"].tolist(),
    "contexts": eval_df["contexts"].tolist(),
    "ground_truth": eval_df["ground_truth"].tolist()
}

ragas_ds = Dataset.from_dict(ragas_dataset)

# Run RAGAS evaluation
try:
    ragas_result = ragas_evaluate(
        ragas_ds,
        metrics=[
            faithfulness,
            answer_relevance,
            context_recall,
            context_precision,
            answer_correctness
        ]
    )
    
    ragas_df = ragas_result.to_pandas()
    print(f"RAGAS Metrics:")
    print(f"  Faithfulness:       {ragas_df['faithfulness'].mean():.3f}")
    print(f"  Answer Relevance:   {ragas_df['answer_relevance'].mean():.3f}")
    print(f"  Context Recall:     {ragas_df['context_recall'].mean():.3f}")
    print(f"  Context Precision:  {ragas_df['context_precision'].mean():.3f}")
    print(f"  Answer Correctness: {ragas_df['answer_correctness'].mean():.3f}")
    
except Exception as e:
    print(f"RAGAS evaluation error (may need more samples): {e}")


In [17]:
df['response'][1]

'\nYes, you can send a money order from USPS as a business. You can fill in whatever you want in the From section of the money order, including your business name and address. The price only includes the money order itself, so you will need to provide an envelope and a stamp if you want to mail it. It is important to keep records of the payment, such as the stub of the money order, and to ask the contractor for a receipt.'